# Keras-native ViT fine-tuning

## Why this notebook exists

The fine-tuned DINOv2 approach uses PyTorch to fine-tune the backbone, which is
arguably non-compliant with the project statement's 'Keras framework for model
implementation' requirement. This notebook implements an unambiguously Keras-compliant
alternative: a Vision Transformer fine-tuned end-to-end using `model.fit()`.

## What's different from the DINOv2 approach

| Aspect | DINOv2 (PyTorch fine-tuning) | ViT (this notebook) |
|---|---|---|
| Backbone | `dinov2-base` | `google/vit-base-patch16-224-in21k` |
| Architecture | 12-layer ViT-base | 12-layer ViT-base (same!) |
| Pretraining | Self-supervised on 142M images | Supervised on ImageNet-21k (14M images) |
| Training framework | PyTorch | **Keras (tf.keras.Model, model.fit)** |
| Keras-compliance | Borderline | Unambiguous |

This lets us do a controlled comparison: the ONLY variable between this experiment
and the DINOv2-base fine-tuning experiment is the pretraining objective.
Any performance difference can be directly attributed to pretraining strategy.

## Approach

HuggingFace's `TFViTModel` is a `tf.keras.Model` subclass — it participates
natively in Keras training, supports `model.fit()`, and can be frozen/unfrozen
using standard Keras mechanisms. We use the same two-phase transfer learning
pattern as the EfficientNetV2S and Xception notebooks:

1. **Phase 1**: Freeze the ViT backbone, train only the classifier head.
   This gets the randomly-initialised head to a reasonable state without
   destabilising the pretrained features.
2. **Phase 2**: Unfreeze the top transformer blocks, continue training end-to-end
   at a much lower learning rate.


## 0 - Imports and config

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import math
import json
from pathlib import Path
import numpy as np

import tensorflow as tf
import keras
from keras import Model, layers
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.utils import image_dataset_from_directory
from keras.backend import clear_session

import tensorflow_addons as tfa

# HuggingFace transformers provides TFViTModel — a tf.keras.Model subclass
# wrapping the Google ViT architecture with pretrained weights.
from transformers import TFViTModel


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:
# GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {[g.name for g in gpus]}')

# XLA compilation for faster training
tf.config.optimizer.set_jit(True)

# Mixed precision: fp16 for compute, fp32 for weights and softmax.
# This is what makes fine-tuning a ViT feasible on 8GB VRAM.
keras.mixed_precision.set_global_policy('mixed_float16')
print(f"Mixed precision policy: {keras.mixed_precision.global_policy()}")


GPU detected: ['/physical_device:GPU:0']
Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0. Your GPU:
  NVIDIA GeForce GTX 1080, compute capability 6.1
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
Mixed precision policy: <Policy "mixed_float16">


In [3]:
# ── Paths ────────────────────────────────────────────────────────────────
DATA_DIR     = Path('../wikiart_split')
CKPT_DIR     = Path('./Checkpoints')
METRICS_DIR  = Path('./Metrics')
CKPT_DIR.mkdir(exist_ok=True)
METRICS_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────
VIT_MODEL   = 'google/vit-base-patch16-224-in21k'
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 16
N_CLASSES   = 23
AUTOTUNE    = tf.data.AUTOTUNE
seed        = 123

# Two-phase training
PHASE1_EPOCHS = 20
PHASE1_LR     = 1e-3
PHASE2_EPOCHS = 15
PHASE2_LR     = 2e-5      # much lower for fine-tuning pretrained weights
N_UNFREEZE    = 4         # of 12 transformer blocks to unfreeze in Phase 2


## 1 - Load datasets

In [4]:
train_ds = image_dataset_from_directory(
    DATA_DIR / 'train',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    DATA_DIR / 'val',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    DATA_DIR / 'test',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

# Load class weights from EDA
with open('../class_weights.json', 'r') as f:
    class_weights = {int(k): v for k, v in json.load(f).items()}


Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


## 2 - Build the ViT classifier

We wrap the HuggingFace `TFViTModel` inside a Keras Functional API model.
`TFViTModel` is itself a `tf.keras.Model` subclass, so it integrates seamlessly
with Keras training — all its weights are visible to `model.fit()`, can be
frozen/unfrozen with `.trainable = False/True`, and save/load like normal Keras weights.

### Input format notes

The ViT model expects:
- Pixel values normalised to [-1, 1] (mean=0.5, std=0.5 per channel)
- Channels-first format: (B, 3, H, W)

Our datasets yield (B, H, W, 3) in [0, 255]. Both transformations are built
into the model's preprocessing layers so the pipeline works end-to-end.

### Feature extraction strategy

Same as the DINOv2 notebook: concatenate the CLS token with the mean of all
patch tokens. This gives a richer 1536-dim representation (768 + 768) than
the CLS token alone.


In [5]:
def build_vit_classifier(vit_backbone, n_classes=N_CLASSES,
                         dropout=0.4, l2_reg=1e-4,
                         image_size=IMAGE_SIZE, name='vit_classifier'):
    """
    Build a Keras classifier on top of a pretrained TFViTModel.

    The backbone is passed in rather than loaded inside, so the same function
    can be used for Phase 1 (backbone frozen) and Phase 2 (backbone trainable)
    without reloading pretrained weights.
    """
    inputs = keras.Input(shape=(*image_size, 3), name='images')

    # Preprocessing: [0, 255] channels-last -> [-1, 1] channels-first
    x = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0, name='rescale')(inputs)
    x = layers.Permute((3, 1, 2), name='to_channels_first')(x)

    # ViT backbone — outputs a BaseModelOutputWithPooling object.
    # .last_hidden_state has shape (B, 197, 768): 1 CLS token + 196 patches (14x14)
    # .pooler_output has shape (B, 768) — CLS after a tanh layer, less informative
    vit_out = vit_backbone(pixel_values=x, training=True).last_hidden_state

    cls_token  = layers.Lambda(lambda t: t[:, 0, :], name='cls_token')(vit_out)
    patch_mean = layers.Lambda(
        lambda t: tf.reduce_mean(t[:, 1:, :], axis=1), name='patch_mean'
    )(vit_out)
    features   = layers.Concatenate(name='cls_patch_concat')([cls_token, patch_mean])

    # Classifier head (mirrors the DINOv2 head for fair comparison)
    h = layers.LayerNormalization(name='feat_norm')(features)
    h = layers.Dense(512, kernel_regularizer=keras.regularizers.l2(l2_reg),
                     name='fc1')(h)
    h = layers.BatchNormalization(name='bn1')(h)
    h = layers.Activation('gelu', name='act1')(h)
    h = layers.Dropout(dropout, name='drop1')(h)

    h = layers.Dense(256, kernel_regularizer=keras.regularizers.l2(l2_reg),
                     name='fc2')(h)
    h = layers.BatchNormalization(name='bn2')(h)
    h = layers.Activation('gelu', name='act2')(h)
    h = layers.Dropout(dropout / 2, name='drop2')(h)

    # Final softmax always in float32 even under mixed precision
    outputs = layers.Dense(n_classes, activation='softmax',
                           dtype='float32', name='head')(h)
    return Model(inputs, outputs, name=name)


def make_metrics(n_classes=N_CLASSES):
    return [
        CategoricalAccuracy(name='accuracy'),
        AUC(multi_label=True, name='auc'),
        tfa.metrics.F1Score(num_classes=n_classes, average='macro', name='f1_score'),
    ]


def cosine_warmup(base_lr, total_epochs, warmup_epochs=3):
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        p = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * p))
    return scheduler


In [6]:
# Load the pretrained ViT backbone once. It will be reused across both phases
# so that Phase 2 starts from Phase 1's head weights plus the pretrained backbone.
print(f'Loading pretrained {VIT_MODEL}...')
vit_backbone = TFViTModel.from_pretrained(VIT_MODEL)
n_params = vit_backbone.count_params()
print(f'ViT backbone: {n_params:,} parameters')
print(f'Number of transformer blocks: {len(vit_backbone.vit.encoder.layer)}')


Loading pretrained google/vit-base-patch16-224-in21k...


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFViTModel.

All the weights of TFViTModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTModel for predictions without further training.


ViT backbone: 86,389,248 parameters
Number of transformer blocks: 12


## 3 - Phase 1: train the classifier head on frozen features

With the backbone frozen, the ViT acts as a fixed feature extractor and only
the Keras head learns. This is fast and gets the randomly-initialised head to
a reasonable starting point before we risk disturbing the pretrained backbone.


In [7]:
# Freeze the entire backbone for Phase 1
vit_backbone.trainable = False

model_p1 = build_vit_classifier(vit_backbone, name='vit_phase1')

trainable_p1 = sum(np.prod(v.shape) for v in model_p1.trainable_variables)
frozen_p1    = sum(np.prod(v.shape) for v in model_p1.non_trainable_variables)
print(f'Phase 1 — trainable: {trainable_p1:,}  frozen: {frozen_p1:,}')


Phase 1 — trainable: 928,791  frozen: 86,390,784


In [8]:
model_p1.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE1_LR, weight_decay=1e-4),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=make_metrics(),
)

callbacks_p1 = [
    ModelCheckpoint(CKPT_DIR / 'ckpt_phase1_vit', monitor='val_loss',
                    save_best_only=True, verbose=0, save_weights_only=False),
    CSVLogger(METRICS_DIR / 'log_phase1_vit.csv'),
    LearningRateScheduler(cosine_warmup(PHASE1_LR, PHASE1_EPOCHS, warmup_epochs=2)),
    EarlyStopping(monitor='val_loss', patience=6,
                  restore_best_weights=True, verbose=1),
]

print('Starting Phase 1: frozen backbone, training head only...')
history_p1 = model_p1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks_p1,
    class_weight=class_weights,
    verbose=1,
)


Starting Phase 1: frozen backbone, training head only...
Epoch 1/20
583/583 [==============================] - ETA: 0s - loss: 2.1656 - accuracy: 0.4738 - auc: 0.8994 - f1_score: 0.4607

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_vit\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_vit\assets


583/583 [==============================] - 189s 279ms/step - loss: 2.1656 - accuracy: 0.4738 - auc: 0.8994 - f1_score: 0.4607 - val_loss: 1.7008 - val_accuracy: 0.6476 - val_auc: 0.9618 - val_f1_score: 0.6334 - lr: 5.0000e-04
Epoch 2/20
583/583 [==============================] - ETA: 0s - loss: 1.6763 - accuracy: 0.6463 - auc: 0.9611 - f1_score: 0.6398

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_vit\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_vit\assets


583/583 [==============================] - 148s 254ms/step - loss: 0.7941 - accuracy: 0.9962 - auc: 1.0000 - f1_score: 0.9966 - val_loss: 1.3221 - val_accuracy: 0.7877 - val_auc: 0.9788 - val_f1_score: 0.7700 - lr: 1.1698e-04
Epoch 18/20
583/583 [==============================] - ETA: 0s - loss: 0.7829 - accuracy: 0.9977 - auc: 1.0000 - f1_score: 0.9981

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_vit\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_vit\assets


583/583 [==============================] - 149s 256ms/step - loss: 0.7829 - accuracy: 0.9977 - auc: 1.0000 - f1_score: 0.9981 - val_loss: 1.3099 - val_accuracy: 0.7922 - val_auc: 0.9795 - val_f1_score: 0.7739 - lr: 6.6987e-05
Epoch 19/20
583/583 [==============================] - 103s 176ms/step - loss: 0.7858 - accuracy: 0.9966 - auc: 1.0000 - f1_score: 0.9969 - val_loss: 1.3154 - val_accuracy: 0.7887 - val_auc: 0.9797 - val_f1_score: 0.7700 - lr: 3.0154e-05
Epoch 20/20
583/583 [==============================] - 104s 178ms/step - loss: 0.8107 - accuracy: 0.9973 - auc: 1.0000 - f1_score: 0.9977 - val_loss: 1.3490 - val_accuracy: 0.7877 - val_auc: 0.9796 - val_f1_score: 0.7700 - lr: 7.5961e-06


In [9]:
# Evaluate Phase 1 result
phase1_results = model_p1.evaluate(test_ds, return_dict=True, verbose=0)
print('Phase 1 (frozen backbone) — test results:')
for k, v in phase1_results.items():
    print(f'  {k}: {v:.4f}')


Phase 1 (frozen backbone) — test results:
  loss: 1.3487
  accuracy: 0.7962
  auc: 0.9795
  f1_score: 0.7810


## 4 - Phase 2: unfreeze top transformer blocks and fine-tune

With the head now at a reasonable starting point, we unfreeze the top N_UNFREEZE
transformer blocks and continue training at a much lower learning rate (2e-5).
The low LR is essential — transformer attention weights are very sensitive to
large gradient updates, and a high LR here would destroy the pretrained features.

We keep the early blocks frozen because they encode generic visual features
(edges, textures, basic shapes) that transfer universally. Only the later blocks
encode semantic information that benefits from adapting to the target domain.


In [10]:
# Unfreeze only the last N_UNFREEZE transformer blocks.
# Start by freezing everything, then selectively unfreeze.
for layer in vit_backbone.vit.encoder.layer:
    layer.trainable = False

total_blocks = len(vit_backbone.vit.encoder.layer)
for layer in vit_backbone.vit.encoder.layer[total_blocks - N_UNFREEZE:]:
    layer.trainable = True

# Also unfreeze the final LayerNorm and pooler (if present)
if hasattr(vit_backbone.vit, 'layernorm'):
    vit_backbone.vit.layernorm.trainable = True

# Set the wrapper model itself to trainable=True.
# Sub-layers remain frozen/unfrozen based on their individual settings.
vit_backbone.trainable = True

# CRITICAL: rebuild the model so the new trainability settings take effect.
# Keras caches trainable status when the model is compiled; we need a fresh
# model referencing the same (partially unfrozen) backbone.
model_p2 = build_vit_classifier(vit_backbone, name='vit_phase2')
# Copy the head weights learned in Phase 1
for src, dst in zip(model_p1.layers, model_p2.layers):
    if 'vit' not in src.name.lower() and src.count_params() > 0:
        try:
            dst.set_weights(src.get_weights())
        except (ValueError, AttributeError):
            pass  # skip layers with different shapes (like the backbone)

trainable_p2 = sum(np.prod(v.shape) for v in model_p2.trainable_variables)
frozen_p2    = sum(np.prod(v.shape) for v in model_p2.non_trainable_variables)
print(f'Phase 2 — trainable: {trainable_p2:,}  frozen: {frozen_p2:,}')
print(f'Unfroze last {N_UNFREEZE}/{total_blocks} transformer blocks')


Phase 2 — trainable: 87,318,039  frozen: 1,536
Unfroze last 4/12 transformer blocks


In [11]:
model_p2.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE2_LR, weight_decay=1e-5),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=make_metrics(),
)

callbacks_p2 = [
    ModelCheckpoint(CKPT_DIR / 'ckpt_phase2_vit', monitor='val_loss',
                    save_best_only=True, verbose=0, save_weights_only=False),
    CSVLogger(METRICS_DIR / 'log_phase2_vit.csv'),
    LearningRateScheduler(cosine_warmup(PHASE2_LR, PHASE2_EPOCHS, warmup_epochs=2)),
    EarlyStopping(monitor='val_loss', patience=6,
                  restore_best_weights=True, verbose=1),
]

print('Starting Phase 2: fine-tuning top transformer blocks...')
history_p2 = model_p2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=callbacks_p2,
    class_weight=class_weights,
    verbose=1,
)


Starting Phase 2: fine-tuning top transformer blocks...
Epoch 1/15


583/583 [==============================] - ETA: 0s - loss: 0.8298 - accuracy: 0.9942 - auc: 1.0000 - f1_score: 0.9950

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_vit\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_vit\assets


583/583 [==============================] - 366s 543ms/step - loss: 0.8298 - accuracy: 0.9942 - auc: 1.0000 - f1_score: 0.9950 - val_loss: 1.2597 - val_accuracy: 0.8072 - val_auc: 0.9836 - val_f1_score: 0.7914 - lr: 1.0000e-05
Epoch 2/15
583/583 [==============================] - ETA: 0s - loss: 0.7771 - accuracy: 0.9973 - auc: 1.0000 - f1_score: 0.9976

...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_vit\assets


583/583 [==============================] - 313s 537ms/step - loss: 0.6993 - accuracy: 0.9999 - auc: 1.0000 - f1_score: 0.9999 - val_loss: 1.1395 - val_accuracy: 0.8559 - val_auc: 0.9890 - val_f1_score: 0.8439 - lr: 1.3546e-05
Epoch 9/15
583/583 [==============================] - ETA: 0s - loss: 0.6940 - accuracy: 1.0000 - auc: 1.0000 - f1_score: 1.0000

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_vit\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_vit\assets


583/583 [==============================] - 311s 533ms/step - loss: 0.6940 - accuracy: 1.0000 - auc: 1.0000 - f1_score: 1.0000 - val_loss: 1.1379 - val_accuracy: 0.8559 - val_auc: 0.9895 - val_f1_score: 0.8411 - lr: 1.1205e-05
Epoch 10/15
583/583 [==============================] - 255s 438ms/step - loss: 0.6901 - accuracy: 1.0000 - auc: 1.0000 - f1_score: 1.0000 - val_loss: 1.1404 - val_accuracy: 0.8584 - val_auc: 0.9892 - val_f1_score: 0.8432 - lr: 8.7946e-06
Epoch 11/15
583/583 [==============================] - 261s 447ms/step - loss: 0.6879 - accuracy: 1.0000 - auc: 1.0000 - f1_score: 1.0000 - val_loss: 1.1424 - val_accuracy: 0.8579 - val_auc: 0.9897 - val_f1_score: 0.8444 - lr: 6.4540e-06
Epoch 12/15
583/583 [==============================] - 256s 440ms/step - loss: 0.6851 - accuracy: 1.0000 - auc: 1.0000 - f1_score: 1.0000 - val_loss: 1.1409 - val_accuracy: 0.8549 - val_auc: 0.9897 - val_f1_score: 0.8411 - lr: 4.3194e-06
Epoch 13/15
583/583 [==============================] - 257s 

## 5 - Evaluate and compare

In [12]:
phase2_results = model_p2.evaluate(test_ds, return_dict=True, verbose=0)
print('Phase 2 (fine-tuned) — test results:')
for k, v in phase2_results.items():
    print(f'  {k}: {v:.4f}')

print()
print('=' * 60)
print('Phase 1 (frozen) vs Phase 2 (fine-tuned)')
print('=' * 60)
print(f"{'Metric':<15} {'Phase 1':>10} {'Phase 2':>10} {'Delta':>8}")
print('-' * 48)
for k in phase1_results:
    if k in phase2_results:
        delta = phase2_results[k] - phase1_results[k]
        sign = '+' if delta >= 0 else ''
        print(f'{k:<15} {phase1_results[k]:>10.4f} {phase2_results[k]:>10.4f} {sign}{delta:>7.4f}')

clear_session()


Phase 2 (fine-tuned) — test results:
  loss: 1.1392
  accuracy: 0.8556
  auc: 0.9901
  f1_score: 0.8424

Phase 1 (frozen) vs Phase 2 (fine-tuned)
Metric             Phase 1    Phase 2    Delta
------------------------------------------------
loss                1.3487     1.1392 -0.2096
accuracy            0.7962     0.8556 + 0.0593
auc                 0.9795     0.9901 + 0.0106
f1_score            0.7810     0.8424 + 0.0614


## Interpreting the result for your report

Three outcomes are possible when comparing this to fine-tuned DINOv2:

**Case 1 — ViT matches or beats DINOv2.** The supervised ImageNet-21k pretraining
turned out to produce features just as useful as DINOv2's self-supervised ones
for this task. Use this as your primary ViT result; it's fully Keras-compliant.

**Case 2 — ViT is close but slightly below DINOv2.** The most likely outcome.
DINOv2's self-supervised objective is designed to cluster images by visual
similarity, which is a better inductive bias for style classification than
ImageNet category labels. Use this ViT result in your report as the primary
ViT experiment, and optionally cite the fine-tuned DINOv2 as 'a non-Keras
extension experiment' — the comparison itself is a good discussion point
(highlights the importance of pretraining objective).

**Case 3 — ViT significantly underperforms DINOv2.** Still useful: report the
ViT as your ViT data point, and dedicate a paragraph in your analysis to
discussing why self-supervised pretraining gives stronger features for
fine-grained style recognition. This is a genuinely interesting finding and
directly addresses the 'critical analysis of results' grading criterion.
